# Práctico 3.2 — Visualización de microbiota con `phyloseq`
**Semana 13 · Bioinformática en el Diagnóstico Molecular**

Trabajaremos con el objeto `Phyloseq_practico.rds` generado en el Práctico 3.1.

**Ruta:** cargar → explorar → composición → diversidad alfa → diversidad beta → comparación exploratoria de géneros. Ejecuta las celdas en orden, usando **R en Google Colab**.

## 1. Instalar paquetes


In [ ]:
install.packages(c("ggplot2", "ggpubr", "dplyr", "tidyr", "vegan"))
if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
BiocManager::install("phyloseq", ask = FALSE, update = FALSE)

### Cargar las librerías

In [ ]:
library(phyloseq)
library(ggplot2)
library(dplyr)
library(tidyr)
library(vegan)
library(ggpubr)

## 2. Cargar el objeto `phyloseq`
Sube `Phyloseq_practico.rds` al panel **Archivos** de Colab.

In [ ]:
ps <- readRDS("Phyloseq_practico.rds")
ps
nsamples(ps)
ntaxa(ps)
sample_variables(ps)
head(as.data.frame(sample_data(ps)))

**Pregunta 1.** ¿Cuántas muestras y ASV contiene el objeto? ¿Qué información almacena `sample_data()`?

## 3. Seleccionar muestras Early y Late
El práctico anterior utilizó los periodos **Early** y **Late**. Excluimos las muestras Mock para comparar solo muestras biológicas. Comprueba que la columna `Periodo` aparece en los metadatos.

In [ ]:
table(sample_data(ps)$Periodo)
ps <- subset_samples(ps, Periodo %in% c("Early", "Late"))
ps <- prune_taxa(taxa_sums(ps) > 0, ps)
ps

**Pregunta 2.** ¿Por qué las muestras Mock no deben mezclarse con las muestras biológicas en esta comparación?

## 4. Abundancia relativa por filo

In [ ]:
ps_rel <- transform_sample_counts(ps, function(x) x / sum(x))
ps_filo <- tax_glom(ps_rel, taxrank = "Phylum")
plot_bar(ps_filo, x = "Periodo", fill = "Phylum") +
  labs(x = "Periodo", y = "Abundancia relativa")

**Pregunta 3.** ¿Qué filos predominan? ¿Por qué las abundancias relativas se expresan como proporciones?

## 5. Diversidad alfa: Observed y Shannon
Los índices se calculan a partir de los conteos originales, no de la tabla transformada a proporciones.

In [ ]:
# Calcular diversidad alfa
alfa <- estimate_richness(ps,
                          measures = c("Observed", "Shannon"))

# Incorporar los metadatos
alfa$Periodo <- as.character(
  sample_data(ps)$Periodo[match(rownames(alfa), sample_names(ps))]
)

# Revisar resultados
head(alfa)
table(alfa$Periodo)

plot_richness(ps, x = "Periodo", measures = c("Observed", "Shannon")) +
  geom_boxplot(alpha = 0.2)

**Pregunta 4.** ¿Qué diferencia existe entre riqueza observada y Shannon? ¿Un gráfico por sí solo demuestra diferencias significativas?

### 5.1 Diversidad alfa con `ggpubr` y valor p

`ggpubr` permite añadir una comparación estadística directamente con `stat_compare_means()`.



In [ ]:
# Boxplot de Shannon
p_shannon <- ggboxplot(alfa, x = "Periodo", y = "Shannon",
                       fill = "Periodo", add = "jitter") +
  labs(x = "Periodo", y = "Índice de Shannon")

  p_shannon <- p_shannon + stat_compare_means()

p_shannon

In [ ]:
# Boxplot de riqueza observada
p_observed <- ggboxplot(alfa, x = "Periodo", y = "Observed",
                        fill = "Periodo", add = "jitter") +
  labs(x = "Periodo", y = "Riqueza observada")

  p_observed <- p_observed + stat_compare_means()

p_observed

**Pregunta 4B.** Compara las medianas de Observed y Shannon. ¿Por qué el gráfico no muestra un valor p por defecto? Si la docente confirma independencia, activa `muestras_independientes <- TRUE` y explica qué indica el valor p. Recuerda que p ≥ 0,05 no demuestra igualdad.

## 6. Diversidad beta: PCoA de Bray–Curtis

In [ ]:
bray <- distance(ps_rel, method = "bray")
pcoa <- ordinate(ps_rel, method = "PCoA", distance = bray)
plot_ordination(ps_rel, pcoa, color = "Periodo") +
  geom_point(size = 3)

**Pregunta 5.** ¿Las muestras Early y Late se separan visualmente? ¿Qué significa que dos puntos estén próximos?

## 7. Visualización exploratoria de géneros bacterianos
**Actividad descriptiva (sin pruebas estadísticas).** Compararemos las abundancias relativas de géneros específicos entre Early y Late. Como los identificadores individuales no permiten establecer pares biológicos confiables, **no aplicaremos Wilcoxon ni hablaremos de abundancia diferencial significativa**.

### 7.1 Agrupar ASV a nivel de género

In [ ]:
ps_genero <- tax_glom(ps, taxrank = "Genus", NArm = TRUE)
ps_genero <- transform_sample_counts(ps_genero, function(x) x / sum(x))
datos <- psmelt(ps_genero)
head(datos[, c("Sample", "Periodo", "Genus", "Abundance")])

### 7.2 Identificar los géneros más abundantes
Seleccionamos un género presente en los datos, en lugar de elegirlo por un valor p.

In [ ]:
generos <- aggregate(Abundance ~ Genus, data = datos, FUN = mean)
generos <- generos[order(generos$Abundance, decreasing = TRUE), ]
head(generos, 10)

### 7.3 Visualizar un género concreto
Seleccionamos el género con mayor abundancia relativa media entre las muestras. El título indicará su nombre.

In [ ]:
genero_ejemplo <- as.character(generos$Genus[1])
datos_genero <- subset(datos, Genus == genero_ejemplo)

ggplot(datos_genero, aes(x = Periodo, y = Abundance)) +
  geom_boxplot(fill = "lightblue") +
  geom_jitter(width = 0.1, size = 2) +
  labs(title = paste("Abundancia relativa de", genero_ejemplo),
       x = "Periodo", y = "Abundancia relativa") +
  theme_classic()

**Pregunta 6.** ¿Qué género se muestra? ¿En qué periodo se observa una mediana mayor? ¿Hay variabilidad entre muestras? ¿Por qué no podemos afirmar una diferencia estadísticamente significativa a partir de este gráfico?

### 7.4 Explorar otro género (opcional)
Reemplaza el número `1` por `2` o `3` para visualizar otro de los géneros más abundantes.

In [ ]:
genero_ejemplo <- as.character(generos$Genus[2])
datos_genero <- subset(datos, Genus == genero_ejemplo)

ggplot(datos_genero, aes(x = Periodo, y = Abundance)) +
  geom_boxplot(fill = "lightgreen") +
  geom_jitter(width = 0.1, size = 2) +
  labs(title = paste("Abundancia relativa de", genero_ejemplo),
       x = "Periodo", y = "Abundancia relativa") +
  theme_classic()

## 8. Cierre del práctico
Redacta 5–7 líneas describiendo: (a) los filos predominantes, (b) diversidad alfa, (c) agrupamiento en la PCoA y (d) el patrón observado en uno de los géneros. **Distingue observación descriptiva de evidencia estadística.**